<a href="https://colab.research.google.com/github/pavanthiriveedi7-rgb/pandas/blob/main/Chaining_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import numpy as np
import pandas as pd

**Before Chaining Methods**


In [28]:
# Load Data
df = pd.read_csv("/content/FAOSTAT_data_en_3-29-2026.csv")

In [29]:
# Step 1: Clean names
df.columns = [x.strip().lower().replace(' ', '_') for x in df.columns]
df

,domain_code,domain,area_code_(m49),area,element_code,element,item_code_(cpc),item,year_code,year,unit,value,flag,flag_description,note
0,QCL,Crops and livestock products,356,India,5312,Area harvested,111,Wheat,2024,2024,ha,31832884,A,Official figure,NaN
1,QCL,Crops and livestock products,356,India,5510,Production,111,Wheat,2024,2024,t,113292368,A,Official figure,NaN


In [30]:
# Step 2: Filter zeros (In-place or reassignment)
df = df[df['value'] > 0]


In [31]:
df

,domain_code,domain,area_code_(m49),area,element_code,element,item_code_(cpc),item,year_code,year,unit,value,flag,flag_description,note
0,QCL,Crops and livestock products,356,India,5312,Area harvested,111,Wheat,2024,2024,ha,31832884,A,Official figure,NaN
1,QCL,Crops and livestock products,356,India,5510,Production,111,Wheat,2024,2024,t,113292368,A,Official figure,NaN


In [32]:
# Step 3: Pivot the table
df_wide = df.pivot_table(index=['area', 'item', 'year'],
                         columns='element',
                         values='value').reset_index()

In [33]:
df_wide

element,area,item,year,Area harvested,Production
0,India,Wheat,2024,31832884.0,113292368.0


In [34]:
# Step 4: Rename for math
df_wide = df_wide.rename(columns={'Production': 'prod', 'Area harvested': 'area_h'})

In [35]:
df_wide

element,area,item,year,area_h,prod
0,India,Wheat,2024,31832884.0,113292368.0


In [36]:
# Step 5: Calculate Median (This requires a separate variable)
median_stats = df.groupby(['area', 'item'])['value'].median().reset_index()
median_stats.columns = ['area', 'item', 'median_val']

In [37]:
median_stats

,area,item,median_val
0,India,Wheat,72562626.0


In [38]:
# Step 6: Merge
df_final = pd.merge(df_wide, median_stats, on=['area', 'item'], how='left')

In [39]:
df_final

,area,item,year,area_h,prod,median_val
0,India,Wheat,2024,31832884.0,113292368.0,72562626.0


In [40]:
# Step 7: Final Math
df_final['area_h'] = df_final['area_h'].fillna(df_final['median_val'])
df_final['yield'] = df_final['prod'] / df_final['area_h']

print(df_final.head())

    area   item  year      area_h         prod  median_val     yield
0  India  Wheat  2024  31832884.0  113292368.0  72562626.0  3.558973


**After Chaining Methods**

In [42]:
import pandas as pd
import numpy as np

# THE PRO PIPELINE
df_agri = (
    pd.read_csv("/content/FAOSTAT_data_en_3-29-2026.csv")

    # 1. Standardize (Vectorized)
    .rename(columns=lambda x: x.strip().lower().replace(' ', '_'))

    # 2. Filter early to save memory
    .query("value > 0")

    # 3. Reshape (Long to Wide)
    .pivot_table(index=['area', 'item', 'year'],
                 columns='element',
                 values='value')
    .reset_index()
    .rename(columns={'Production': 'prod', 'Area harvested': 'area_h'})

    # 4. Self-Merge (Using a lambda to reference the 'living' dataframe)
    .merge(
        (pd.read_csv("/content/FAOSTAT_data_en_3-29-2026.csv")
         .groupby(['Area', 'Item'])['Value'].median().reset_index()
         .rename(columns={'Area': 'area', 'Item': 'item', 'Value': 'med_area'})),
        on=['area', 'item'],
        how='left'
    )

    # 5. Final Mathematical Transformation
    .assign(
        area_h = lambda x: x['area_h'].fillna(x['med_area']),
        calc_yield = lambda x: (x['prod'] / x['area_h']).replace([np.inf, -np.inf], 0)
    )

    # 6. Logical Filter
    .loc[lambda x: x['calc_yield'] > 0]
)

print(df_agri.head())

    area   item  year      area_h         prod    med_area  calc_yield
0  India  Wheat  2024  31832884.0  113292368.0  72562626.0    3.558973
